In [8]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import open3d as o3d

from tqdm import tqdm

import configs as cfg

In [4]:
db_mesh_folder = Path(cfg.working_directory) / "outputs" / "models"

ms_csv_path = Path(cfg.working_directory) / "outputs" / "mesh_area_volume.csv"

In [6]:
mesh_csv = pd.read_csv( ms_csv_path , dtype={'label': str})

mesh_csv.head()

,label,area (cm2),volume (cm3)
0,000,156.907144,155.142952
1,001,196.060157,232.677661
2,002,285.878568,385.034387
3,003,217.144428,276.909959
4,004,215.936410,261.357869


In [10]:
results = []

for index, row in tqdm( mesh_csv.iterrows(), total=len(mesh_csv), desc="Processing meshes" ):
    mesh_path = db_mesh_folder / row['label'] / f"{row['label']}.obj"

    if not mesh_path.exists():
        print(f"{row['label']} mesh file not exists")
        continue

    mesh = o3d.io.read_triangle_mesh(mesh_path)
    bbox = mesh.get_axis_aligned_bounding_box()
    bbox_extent = bbox.get_extent()
    
    # 排序三个轴长
    sorted_extent = np.sort(bbox_extent)
    aspect_ratio = sorted_extent[2] / sorted_extent[0]  # 最长/最短轴比

    # 2. 获取表面积和体积
    surface_area = row['area (cm2)']
    volume = row['volume (cm3)']

    # 3. 计算球形度指数
    sphere_surface = (36 * np.pi * volume**2) ** (1/3)  # 等效球体表面积
    sphericity = sphere_surface / surface_area

    # 4. 计算体积表面积比
    volume_surface_ratio = volume / surface_area

    # 计算凸包
    convex_hull = mesh.compute_convex_hull()[0]  # 返回(TriangleMesh, vertex_indices)
    
    # 计算凸包体积
    convex_volume = convex_hull.get_volume() * 1000 * 1000
    convexity = volume / convex_volume

    # 存储结果
    results.append({
        'label': row['label'],
        'minor axis length (cm)': bbox_extent[0] * 100,
        'middle axis length (cm)': bbox_extent[1] * 100,
        'major axis length (cm)': bbox_extent[2] * 100,
        'area (cm2)': surface_area,
        'volume (cm3)': volume,
        'aspect ratio': aspect_ratio,
        'volume/surface ratio': volume_surface_ratio,
        'sphericity': sphericity,
        'convexity': convexity
    })

results_pd = pd.DataFrame(results)

results_pd.head()

Processing meshes: 100%|██████████| 230/230 [01:57<00:00,  1.95it/s]


,label,minor axis length (cm),middle axis length (cm),major axis length (cm),area (cm2),volume (cm3),aspect ratio,volume/surface ratio,sphericity,convexity
0,000,6.3611,5.1731,10.0520,156.907144,155.142952,1.943129,0.988756,0.889876,0.925711
1,001,7.7410,9.4129,7.0252,196.060157,232.677661,1.339876,1.186767,0.933107,0.970746
2,002,7.1185,13.5962,8.6997,285.878568,385.034387,1.909981,1.346846,0.895301,0.960785
3,003,9.7218,7.6250,7.9210,217.144428,276.909959,1.274990,1.275234,0.946153,0.976820
4,004,6.5068,11.5171,7.5362,215.936410,261.357869,1.770010,1.210346,0.915480,0.960588


In [11]:
results_pd.to_csv( Path(cfg.working_directory) / "outputs" / "mesh_traits.csv", index=None)